## 11 — Visualisations: pressure dynamics, shot quality, model predictions

Generates all analytical plots. Reads Gold data and M3 model results from notebook 10 (model must be fitted in the same session or results available in context).

Plots:
1. **Pressure + xG for a sample match** — time series
2. **Predicted shot probability vs pressure advantage** — main model result
3. **Pressure dynamics before a shot** — event study (10 minutes before shot)
4. **Pressure trajectories: shot vs no-shot minutes** — group comparison
5. **Forest plot** — ORs with CIs for M3
6. **Heatmap** — P(shot) as a function of match time and pressure advantage
7. **Pressure trajectories by shot quality** — low/medium/high xG


### 1. Imports 

`expit` (logistic function) from scipy is used to convert log-odds → probability. `build_design_matrices` from patsy generates design matrices for new data based on the model formula.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from patsy import build_design_matrices

### 2. Load Gold momentum 

Reads `gold.match_momentum` and converts to pandas. Selects only needed columns to limit data transfer size.


In [ ]:
momentum_pd = (
    spark.table("wsl_analytics.gold.match_momentum")
    .select(
        "sportmonks_fixture_id",
        "fotmob_match_id",
        "sportmonks_team_id",
        "sportmonks_team_name",
        "location",
        "minute",
        "pressure",
        "shot_count",
        "shot_xg",
        "max_shot_xg",
        "shot_xgot",
        "shots_on_target",
        "goal_count"
    )
    .toPandas()
)

### 3. Preprocess 

Fills NA in shot columns with 0, creates `has_shot` and `has_goal` flags, builds `team_match_id`, and sorts chronologically.


In [ ]:
for col in [
    "shot_count",
    "shot_xg",
    "shot_xgot",
    "shots_on_target",
    "goal_count"
]:
    momentum_pd[col] = momentum_pd[col].fillna(0)

momentum_pd["has_shot"] = (
    momentum_pd["shot_count"] > 0
).astype(int)

momentum_pd["has_goal"] = (
    momentum_pd["goal_count"] > 0
).astype(int)

momentum_pd["team_match_id"] = (
    momentum_pd["sportmonks_fixture_id"].astype(str)
    + "_"
    + momentum_pd["sportmonks_team_id"].astype(str)
)

momentum_pd = momentum_pd.sort_values(
    [
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "minute"
    ]
)

### 4. Add opponent pressure 

Self-merge on (`sportmonks_fixture_id`, `minute`) excluding the same `sportmonks_team_id` — adds opponent pressure to each row. Needed for comparative plots.


In [ ]:
opp = (
    momentum_pd[
        [
            "sportmonks_fixture_id",
            "minute",
            "sportmonks_team_id",
            "pressure"
        ]
    ]
    .rename(
        columns={
            "sportmonks_team_id": "opponent_team_id",
            "pressure": "opponent_pressure"
        }
    )
)

momentum_with_opp = momentum_pd.merge(
    opp,
    on=[
        "sportmonks_fixture_id",
        "minute"
    ],
    how="left"
)

momentum_with_opp = (
    momentum_with_opp[
        momentum_with_opp["sportmonks_team_id"]
        !=
        momentum_with_opp["opponent_team_id"]
    ]
    .copy()
)

### 5. Select example fixture 

Selects the match with the highest total xG — will be the representative example for the time series plot.


In [ ]:
example_fixture = (
    momentum_pd
    .groupby("sportmonks_fixture_id")["shot_xg"]
    .sum()
    .idxmax()
)

print("Fixture:", example_fixture)

### 6. Plot 1: Pressure + xG time series 

Dual Y-axis plot: left axis (pressure), right axis (shot xG). Shots as scatter plot with point size proportional to xG. Goals marked with a star. Vertical dashed line at minute 45 (half-time).


In [ ]:
match = momentum_pd[
    momentum_pd["sportmonks_fixture_id"]
    == example_fixture
].copy()

fig, ax1 = plt.subplots(
    figsize=(16, 7)
)

# PRESSURE
for team, team_df in match.groupby(
    "sportmonks_team_name"
):
    ax1.plot(
        team_df["minute"],
        team_df["pressure"],
        linewidth=2,
        label=f"{team} – pressure"
    )

ax1.set_xlabel("Minute")
ax1.set_ylabel("Pressure Index")
ax1.axvline(45, linestyle="--", alpha=.4)

# xG / shots
ax2 = ax1.twinx()

shots = match[
    match["shot_count"] > 0
]

ax2.scatter(
    shots["minute"],
    shots["shot_xg"],
    s=50 + shots["shot_xg"] * 500,
    alpha=.7,
    label="Shot – size = xG"
)

goals = shots[
    shots["has_goal"] == 1
]

ax2.scatter(
    goals["minute"],
    goals["shot_xg"],
    marker="*",
    s=250,
    label="Goal"
)

ax2.set_ylabel("Shot xG")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper right"
)

ax1.set_title(
    f"Pressure and shot creation – fixture {example_fixture}"
)

plt.tight_layout()
plt.show()

### 7. Reload + filter for modeling 

Reads `gold.match_momentum_temporal` and applies the same filters as notebook 10 — ensures consistency with the model.


In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

In [ ]:
temporal_df = spark.table(
    "wsl_analytics.gold.match_momentum_temporal"
)

modeling_spark_df = (
    temporal_df

    .filter(
        F.col("temporal_prev_5m_n") == 5
    )

    .filter(
        F.col("pressure_prev_5m_n") == 5
    )

    .filter(
        F.col("opponent_pressure_prev_5m_n") == 5
    )

    .filter(
        F.col("target_shot").isNotNull()
    )

    .select(
        "team_match_id",

        "sportmonks_fixture_id",
        "sportmonks_team_id",

        "location",
        "minute",

        "target_shot",

        "shot_lag_1",
        "shots_prev_3m_count",
        "shots_prev_5m_count",
        "xg_prev_5m",

        "pressure_prev_5m_mean",
        "pressure_prev_5m_max",
        "pressure_change_5m",

        "opponent_pressure_prev_5m_mean",
        "pressure_advantage_prev_5m"
    )
)

### 8. Convert to pandas 

Converts with sorting by `(team_match_id, minute)`.


In [ ]:
model_df = (
    modeling_spark_df
    .toPandas()
    .sort_values(
        [
            "team_match_id",
            "minute"
        ]
    )
    .reset_index(drop=True)
)

### 9. Shape check 

Prints dimensions of the modeling DataFrame.


In [ ]:
model_df.shape

### 10. Standardise 

Standardises pressure variables — same steps as notebook 10.


In [ ]:
pressure_variables = [
    "pressure_prev_5m_mean",
    "pressure_prev_5m_max",
    "pressure_change_5m",
    "opponent_pressure_prev_5m_mean",
    "pressure_advantage_prev_5m"
]

for var in pressure_variables:

    mean = model_df[var].mean()
    sd = model_df[var].std()

    model_df[f"{var}_z"] = (
        model_df[var] - mean
    ) / sd

### 11. Describe standardised 

Descriptive statistics after standardisation.


In [ ]:
model_df[
    [
        "pressure_prev_5m_mean_z",
        "opponent_pressure_prev_5m_mean_z",
        "pressure_change_5m_z",
        "pressure_advantage_prev_5m_z"
    ]
].describe()

### 12. Install + import statsmodels

Installs and imports statsmodels.


In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

### 13. M3 formula 

M3 formula — same as in notebook 10, must be re-defined in this notebook.


In [ ]:
formula_3 = """
target_shot ~
    minute
    + I(minute ** 2)
    + C(location)
    + C(sportmonks_team_id)

    + shot_lag_1
    + shots_prev_5m_count
    + xg_prev_5m

    + pressure_advantage_prev_5m_z
    + pressure_change_5m_z
"""

### 14. Fit M3 

Fits M3 model. `result_3` will be used for all predictions.


In [ ]:
model_3 = smf.gee(
    formula=formula_3,

    groups="team_match_id",

    time="minute",

    data=model_df,

    family=sm.families.Binomial(),

    cov_struct=sm.cov_struct.Autoregressive(
        grid=True
    )
)

result_3 = model_3.fit(
    maxiter=300,
    cov_type="robust"
)

print("Converged:", result_3.converged)
print("AR(1):", result_3.cov_struct.dep_params)

### 15. Build prediction grid 

Creates a grid of pressure advantage values from -2.5 to +2.5 SD, holding all other variables at reference values (median minute, home, no previous shots, reference team_id). This isolates the pressure advantage effect.


In [ ]:
reference_team = (
    model_df["sportmonks_team_id"]
    .mode()
    .iloc[0]
)

pressure_grid = np.linspace(
    -2.5,
    2.5,
    101
)

prediction_df = pd.DataFrame({
    "pressure_advantage_prev_5m_z": pressure_grid,

    "pressure_change_5m_z": 0,

    "minute": model_df["minute"].median(),

    "location": "home",

    "sportmonks_team_id": reference_team,

    "shot_lag_1": 0,

    "shots_prev_5m_count": 0,

    "xg_prev_5m": 0
})

### 16. Compute predictions with CI 

Uses `build_design_matrices` from patsy to build the design matrix for new data (preserving category encoding from the original model). Computes log-odds `eta`, standard error `eta_se` via quadratic form of the covariance matrix, then converts to probability via `expit`. Confidence intervals are constructed on the log-odds scale before transformation (delta method).


In [ ]:
design_info = (
    result_3
    .model
    .data
    .design_info
)

X_new = build_design_matrices(
    [design_info],
    prediction_df,
    return_type="dataframe"
)[0]

beta = result_3.params.values
cov = result_3.cov_params().values

eta = X_new.values @ beta

eta_se = np.sqrt(
    np.einsum(
        "ij,jk,ik->i",
        X_new.values,
        cov,
        X_new.values
    )
)

prediction_df["prob"] = expit(eta)

prediction_df["lower"] = expit(
    eta - 1.96 * eta_se
)

prediction_df["upper"] = expit(
    eta + 1.96 * eta_se
)

### 17. Plot 2: P(shot) vs pressure advantage 

Main model plot: predicted shot probability as a function of 5-minute pressure advantage (in SDs). Shaded 95% CI. Vertical dashed line at 0 (no pressure advantage).


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    prediction_df["pressure_advantage_prev_5m_z"],
    prediction_df["prob"],
    linewidth=2.5
)

ax.fill_between(
    prediction_df["pressure_advantage_prev_5m_z"],
    prediction_df["lower"],
    prediction_df["upper"],
    alpha=.2
)

ax.axvline(
    0,
    linestyle="--",
    alpha=.5
)

ax.set_xlabel(
    "Pressure advantage in previous 5 minutes (SD)"
)

ax.set_ylabel(
    "Predicted probability of shot"
)

ax.set_title(
    "Pressure advantage predicts a shot in the next minute"
)

plt.tight_layout()
plt.show()

### 18. Prepare event study data 

Builds the event set (shots) and pressure history in the [-10, -1] minute window before each shot. `relative_minute` is time relative to the event.


In [ ]:
shot_events = (
    momentum_with_opp[
        momentum_with_opp["has_shot"] == 1
    ]
    [
        [
            "sportmonks_fixture_id",
            "sportmonks_team_id",
            "team_match_id",
            "minute",
            "shot_xg",
            "max_shot_xg"
        ]
    ]
    .rename(
        columns={
            "minute": "shot_minute"
        }
    )
)

In [ ]:
history = momentum_with_opp[
    [
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "team_match_id",
        "minute",
        "pressure",
        "opponent_pressure"
    ]
]

In [ ]:
event_df = shot_events.merge(
    history,
    on=[
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "team_match_id"
    ],
    how="left"
)

event_df["relative_minute"] = (
    event_df["minute"]
    -
    event_df["shot_minute"]
)

event_df = event_df[
    event_df["relative_minute"].between(
        -10,
        -1
    )
]

### 19. Aggregate event study 

Two-stage aggregation: first mean for `(team_match_id, relative_minute)`, then grand mean over `relative_minute`. The first stage prevents matches with many shots from dominating.


In [ ]:
event_cluster = (
    event_df
    .groupby(
        [
            "team_match_id",
            "relative_minute"
        ]
    )
    [
        [
            "pressure",
            "opponent_pressure"
        ]
    ]
    .mean()
    .reset_index()
)

In [ ]:
event_summary = (
    event_cluster
    .groupby("relative_minute")
    .agg(
        pressure_mean=(
            "pressure",
            "mean"
        ),
        pressure_se=(
            "pressure",
            "sem"
        ),
        opponent_mean=(
            "opponent_pressure",
            "mean"
        ),
        opponent_se=(
            "opponent_pressure",
            "sem"
        )
    )
    .reset_index()
)

### 20. Plot 3: Pressure dynamics before shot 

Event study: mean pressure of the attacking team and its opponent in the 10 minutes before a shot. Shaded 95% CI (1.96 × SE).


In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 6)
)

ax.plot(
    event_summary["relative_minute"],
    event_summary["pressure_mean"],
    marker="o",
    label="Attacking team"
)

ax.fill_between(
    event_summary["relative_minute"],
    event_summary["pressure_mean"]
    - 1.96 * event_summary["pressure_se"],
    event_summary["pressure_mean"]
    + 1.96 * event_summary["pressure_se"],
    alpha=.2
)

ax.plot(
    event_summary["relative_minute"],
    event_summary["opponent_mean"],
    marker="o",
    label="Opponent"
)

ax.fill_between(
    event_summary["relative_minute"],
    event_summary["opponent_mean"]
    - 1.96 * event_summary["opponent_se"],
    event_summary["opponent_mean"]
    + 1.96 * event_summary["opponent_se"],
    alpha=.2
)

ax.axvline(
    0,
    linestyle="--"
)

ax.set_xlabel(
    "Minutes before shot"
)

ax.set_ylabel(
    "Mean Pressure Index"
)

ax.set_title(
    "Pressure dynamics before a shot"
)

ax.legend()

plt.tight_layout()
plt.show()

### 21. Prepare comparison data   

Builds pressure lags 1–10 for all rows to compare pressure trajectories between shot minutes (`has_shot == 1`) and no-shot minutes (`has_shot == 0`).


In [ ]:
history_compare = (
    momentum_with_opp
    .sort_values(
        [
            "team_match_id",
            "minute"
        ]
    )
    .copy()
)

In [ ]:
for lag in range(1, 11):

    history_compare[
        f"pressure_lag_{lag}"
    ] = (
        history_compare
        .groupby("team_match_id")["pressure"]
        .shift(lag)
    )

### 22. Melt to long format 

Melts pressure lags to long format for aggregation.


In [ ]:
pressure_lag_columns = [
    f"pressure_lag_{i}"
    for i in range(1, 11)
]

event_comparison_long = (
    history_compare
    .melt(
        id_vars=[
            "team_match_id",
            "has_shot"
        ],
        value_vars=pressure_lag_columns,
        var_name="lag",
        value_name="pressure_before"
    )
)

### 23. Extract relative minute 

Extracts the lag number from the column name (`pressure_lag_N`) and converts to negative value (N minutes before the event).


In [ ]:
event_comparison_long[
    "relative_minute"
] = -(
    event_comparison_long["lag"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)

### 24. Aggregate and plot trajectories 

Aggregates and plots Chart 4: mean pressure trajectories for shot vs no-shot minutes (with 95% CI). The difference between groups is the main visual argument for the pressure effect.


In [ ]:
comparison_cluster = (
    event_comparison_long
    .groupby(
        [
            "team_match_id",
            "has_shot",
            "relative_minute"
        ]
    )["pressure_before"]
    .mean()
    .reset_index()
)

In [ ]:
comparison_summary = (
    comparison_cluster
    .groupby(
        [
            "has_shot",
            "relative_minute"
        ]
    )
    .agg(
        mean_pressure=(
            "pressure_before",
            "mean"
        ),
        se=(
            "pressure_before",
            "sem"
        )
    )
    .reset_index()
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 6)
)

labels = {
    0: "No shot at t",
    1: "Shot at t"
}

for shot_value in [0, 1]:

    temp = comparison_summary[
        comparison_summary["has_shot"]
        == shot_value
    ]

    ax.plot(
        temp["relative_minute"],
        temp["mean_pressure"],
        marker="o",
        label=labels[shot_value]
    )

    ax.fill_between(
        temp["relative_minute"],
        temp["mean_pressure"]
        - 1.96 * temp["se"],
        temp["mean_pressure"]
        + 1.96 * temp["se"],
        alpha=.15
    )

ax.set_xlabel(
    "Minutes before target minute"
)

ax.set_ylabel(
    "Mean Pressure Index"
)

ax.set_title(
    "Pressure trajectories before shot vs no-shot minutes"
)

ax.legend()

plt.tight_layout()
plt.show()

### 25. Forest plot data 

Builds a DataFrame with parameters, ORs, and CIs for the forest plot.


In [ ]:
ci = result_3.conf_int()

forest_df = pd.DataFrame({
    "term": result_3.params.index,
    "B": result_3.params.values,
    "OR": np.exp(
        result_3.params.values
    ),
    "low": np.exp(
        ci.iloc[:, 0].values
    ),
    "high": np.exp(
        ci.iloc[:, 1].values
    ),
    "p": result_3.pvalues.values
})

### 26. Term labels 

Dictionary mapping technical model term names to readable plot labels.


In [ ]:
terms = {
    "C(location)[T.home]":
        "Home",

    "shot_lag_1":
        "Shot in previous minute",

    "shots_prev_5m_count":
        "Shots in previous 5 min",

    "xg_prev_5m":
        "xG in previous 5 min",

    "pressure_advantage_prev_5m_z":
        "Pressure advantage",

    "pressure_change_5m_z":
        "Pressure increase"
}

### 27. Plot 5: Forest plot 

Forest plot of ORs with 95% CIs for M3. X-axis on log scale (log-OR is additive, OR is multiplicative). Dashed line at OR=1 (null effect).


In [ ]:
forest_plot = (
    forest_df[
        forest_df["term"].isin(
            terms.keys()
        )
    ]
    .copy()
)

forest_plot["label"] = (
    forest_plot["term"]
    .map(terms)
)

forest_plot = forest_plot.sort_values(
    "OR"
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

y = np.arange(
    len(forest_plot)
)

ax.errorbar(
    forest_plot["OR"],
    y,

    xerr=[
        forest_plot["OR"]
        - forest_plot["low"],

        forest_plot["high"]
        - forest_plot["OR"]
    ],

    fmt="o",
    capsize=4
)

ax.axvline(
    1,
    linestyle="--"
)

ax.set_yticks(y)

ax.set_yticklabels(
    forest_plot["label"]
)

ax.set_xscale("log")

ax.set_xlabel(
    "Odds Ratio (95% CI)"
)

ax.set_title(
    "Predictors of a shot in the next minute"
)

plt.tight_layout()
plt.show()

### 28. Heatmap grid 

Builds a prediction grid over two dimensions: match minute (5–100 in steps of 5) and pressure advantage (-2 to +2 SD).


In [ ]:
minutes_grid = np.arange(
    5,
    101,
    5
)

pressure_values = np.linspace(
    -2,
    2,
    21
)

In [ ]:
heatmap_grid = pd.DataFrame(
    [
        {
            "minute": minute,
            "pressure_advantage_prev_5m_z": pressure,

            "pressure_change_5m_z": 0,

            "location": "home",

            "sportmonks_team_id":
                reference_team,

            "shot_lag_1": 0,

            "shots_prev_5m_count": 0,

            "xg_prev_5m": 0
        }

        for minute in minutes_grid

        for pressure in pressure_values
    ]
)

### 29. Compute heatmap predictions 

Computes predicted P(shot) for the entire grid and pivots to wide format.


In [ ]:
X_heat = build_design_matrices(
    [design_info],
    heatmap_grid,
    return_type="dataframe"
)[0]

heatmap_grid["prob"] = expit(
    X_heat.values
    @
    result_3.params.values
)

In [ ]:
heat = (
    heatmap_grid
    .pivot(
        index="minute",
        columns="pressure_advantage_prev_5m_z",
        values="prob"
    )
)

### 30. Plot 6: Heatmap 

Heatmap of P(shot) as a function of match minute (Y-axis) and pressure advantage (X-axis). Shows whether the pressure effect is stronger at certain stages of the match (e.g. late in the game).


In [ ]:
fig, ax = plt.subplots(
    figsize=(13, 7)
)

im = ax.imshow(
    heat.values,
    aspect="auto",
    origin="lower"
)

ax.set_xticks(
    np.arange(
        len(heat.columns)
    )[::4]
)

ax.set_xticklabels(
    [
        f"{x:.1f}"
        for x in heat.columns[::4]
    ]
)

ax.set_yticks(
    np.arange(
        len(heat.index)
    )
)

ax.set_yticklabels(
    heat.index
)

ax.set_xlabel(
    "Pressure advantage (SD)"
)

ax.set_ylabel(
    "Minute"
)

ax.set_title(
    "Predicted probability of shot by match time and Pressure advantage"
)

fig.colorbar(
    im,
    ax=ax,
    label="Predicted P(shot)"
)

plt.tight_layout()
plt.show()

### 31. xG groups 

Categorises shots by xG quality: low (<0.10), medium (0.10–0.30), high (>0.30).


In [ ]:
event_df["xg_group"] = pd.cut(
    event_df["max_shot_xg"],

    bins=[
        -np.inf,
        0.10,
        0.30,
        np.inf
    ],

    labels=[
        "Low xG (< .10)",
        "Medium xG (.10–.30)",
        "High xG (> .30)"
    ]
)

### 32. Aggregate + Plot 7: Pressure by xG quality 

Aggregates mean pressure in the 10 minutes before shots grouped by xG category and plots Chart 7. Question: is the pressure build-up before high-xG shots stronger than before low-quality shots?


In [ ]:
xg_cluster = (
    event_df
    .groupby(
        [
            "team_match_id",
            "xg_group",
            "relative_minute"
        ],
        observed=True
    )["pressure"]
    .mean()
    .reset_index()
)

In [ ]:
xg_summary = (
    xg_cluster
    .groupby(
        [
            "xg_group",
            "relative_minute"
        ],
        observed=True
    )
    .agg(
        mean_pressure=(
            "pressure",
            "mean"
        ),
        se=(
            "pressure",
            "sem"
        )
    )
    .reset_index()
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 6)
)

for group in [
    "Low xG (< .10)",
    "Medium xG (.10–.30)",
    "High xG (> .30)"
]:

    temp = xg_summary[
        xg_summary["xg_group"]
        == group
    ]

    ax.plot(
        temp["relative_minute"],
        temp["mean_pressure"],
        marker="o",
        label=group
    )

    ax.fill_between(
        temp["relative_minute"],
        temp["mean_pressure"]
        - 1.96 * temp["se"],

        temp["mean_pressure"]
        + 1.96 * temp["se"],

        alpha=.12
    )

ax.set_xlabel(
    "Minutes before shot"
)

ax.set_ylabel(
    "Mean Pressure Index"
)

ax.set_title(
    "Pressure trajectories before shots of different quality"
)

ax.legend()

plt.tight_layout()
plt.show()